# Physics Loss — Step-by-Step Understanding

This notebook explains **how `physics_loss` works** by building up from the simplest possible case.

We go through:
1. What `requires_grad` does
2. First derivative with `autograd.grad`
3. Second derivative (the diagonal of the Hessian)
4. The full Laplacian on a toy example
5. The full Laplacian with ThermalNet on fake data

In [36]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import numpy as np
from arch import ThermalNet

torch.manual_seed(0)
print('PyTorch version:', torch.__version__)

PyTorch version: 2.7.1


---
## PART 1 — What does `requires_grad_(True)` do?

Normally PyTorch just computes numbers. When you set `requires_grad=True`,
it **records every operation** on that tensor so it can later compute derivatives.

In [37]:
# Without requires_grad — just a number, no tracking
x_plain = torch.tensor(3.0)
y_plain = x_plain ** 2
print('y =', y_plain.item())
print('grad_fn:', y_plain.grad_fn)   # None — nothing was tracked

y = 9.0
grad_fn: None


In [38]:
# With requires_grad — PyTorch tracks the computation
x_tracked = torch.tensor(3.0, requires_grad=True)
y_tracked = x_tracked ** 2
print('y =', y_tracked.item())
print('grad_fn:', y_tracked.grad_fn)   # PowBackward — it knows how to differentiate

y = 9.0
grad_fn: <PowBackward0 object at 0x7acbe8ee1d20>


---
## PART 2 — First derivative with `autograd.grad`

Let's use the simplest possible function:  `T = x²`

Analytic answer:  `dT/dx = 2x`

In [39]:
x = torch.tensor(3.0, requires_grad=True)
T = x ** 2          # T = x²

# Compute dT/dx
g = torch.autograd.grad(
    T,              # output to differentiate
    x,              # differentiate w.r.t. this
    create_graph=True  # keep graph so we can differentiate again later
)[0]

print(f'x     = {x.item()}')
print(f'T     = x² = {T.item()}')
print(f'dT/dx = 2x = {g.item()}    (expected: {2*3.0})')

x     = 3.0
T     = x² = 9.0
dT/dx = 2x = 6.0    (expected: 6.0)


---
## PART 3 — Second derivative

We differentiate `g = dT/dx` once more w.r.t. `x`.

Analytic answer:  `d²T/dx² = 2`

In [40]:
# g was computed above: g = dT/dx = 2x
# Now differentiate g w.r.t. x again
H = torch.autograd.grad(
    g,              # differentiate THIS (which is dT/dx)
    x,              # w.r.t. x again
    create_graph=True
)[0]

print(f'dT/dx  = 2x = {g.item()}')
print(f'd²T/dx² = 2  = {H.item()}    (expected: 2.0)')

dT/dx  = 2x = 6.0
d²T/dx² = 2  = 2.0    (expected: 2.0)


---
## PART 4 — Multiple points at once (batch)

In practice we have N points. We can't call `autograd.grad` on a vector directly —
we call it on `T.sum()` (a scalar).

**Why does `.sum()` work?**  
Each point is independent: `d(T_0 + T_1 + T_2)/dx_0 = dT_0/dx_0`  
The sum gradient is just the per-point gradient stacked.

In [41]:
# 5 independent points
x_batch = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0], requires_grad=True)  # shape (5,)
T_batch = x_batch ** 2    # T_i = x_i²

print('x_batch:', x_batch.detach().numpy())
print('T_batch:', T_batch.detach().numpy())

# First derivative: dT_i/dx_i = 2*x_i
g_batch = torch.autograd.grad(T_batch.sum(), x_batch, create_graph=True)[0]
print('\ndT/dx (expected 2x):', g_batch.detach().numpy())
print('2*x_batch:           ', (2 * x_batch).detach().numpy())

# Second derivative: d²T_i/dx_i² = 2
H_batch = torch.autograd.grad(g_batch.sum(), x_batch, create_graph=True)[0]
print('\nd²T/dx² (expected 2.0 everywhere):', H_batch.detach().numpy())

x_batch: [1. 2. 3. 4. 5.]
T_batch: [ 1.  4.  9. 16. 25.]

dT/dx (expected 2x): [ 2.  4.  6.  8. 10.]
2*x_batch:            [ 2.  4.  6.  8. 10.]

d²T/dx² (expected 2.0 everywhere): [2. 2. 2. 2. 2.]


---
## PART 5 — 3D inputs: the Hessian diagonal

Now we have points with 3 spatial coordinates: `(x, y, z)`.

Let's use:  `T = x² + y² + z²`

Analytic answers:
- `∂T/∂x = 2x`,  `∂T/∂y = 2y`,  `∂T/∂z = 2z`
- `∂²T/∂x² = 2`,  `∂²T/∂y² = 2`,  `∂²T/∂z² = 2`
- `∇²T = 2 + 2 + 2 = 6`  (NOT zero — this function does NOT satisfy Laplace!)

The key point: `autograd.grad` returns a gradient for **all input columns**.
We pick column `j` to get the diagonal term `∂²T/∂x_j²`.

In [59]:
# 3 points, each with (x, y, z) coordinates
pts = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
], requires_grad=True)   # shape (3, 3)

# T = x² + y² + z²   for each point
T = (pts ** 2).sum(dim=1, keepdim=True)   # shape (3, 1)
print('T = x²+y²+z²:', T.detach().numpy().flatten())

# ── FIRST DERIVATIVES ────────────────────────────────────────────────────────
# autograd.grad returns shape (3, 3): for each point, gradient w.r.t. each column
g = torch.autograd.grad(T.sum(), pts, create_graph=True)[0]   # (3, 3)

print('\nFirst derivatives (should be 2*pts):')
print('  g[:, 0] = ∂T/∂x:', g[:, 0].detach().numpy(), '  expected:', (2*pts[:, 0]).detach().numpy())
print('  g[:, 1] = ∂T/∂y:', g[:, 1].detach().numpy(), '  expected:', (2*pts[:, 1]).detach().numpy())
print('  g[:, 2] = ∂T/∂z:', g[:, 2].detach().numpy(), '  expected:', (2*pts[:, 2]).detach().numpy())

T = x²+y²+z²: [ 14.  77. 194.]

First derivatives (should be 2*pts):
  g[:, 0] = ∂T/∂x: [ 2.  8. 14.]   expected: [ 2.  8. 14.]
  g[:, 1] = ∂T/∂y: [ 4. 10. 16.]   expected: [ 4. 10. 16.]
  g[:, 2] = ∂T/∂z: [ 6. 12. 18.]   expected: [ 6. 12. 18.]


In [60]:
# ── SECOND DERIVATIVES (diagonal of Hessian) ─────────────────────────────────
print('Second derivatives (should all be 2.0):')

lap = torch.zeros(3)   # accumulate Laplacian for each point

for j in range(3):
    direction = ['x', 'y', 'z'][j]

    # Differentiate g[:, j] w.r.t. pts → shape (3, 3)
    full_grad = torch.autograd.grad(g[:, j].sum(), pts, create_graph=True)[0]

    print(f'\n  Differentiating ∂T/∂{direction} again w.r.t. pts:')
    print(f'  full_grad shape: {full_grad.shape}  (one row per point, one col per input dim)')
    print(f'  full_grad =\n{full_grad.detach().numpy().round(2)}')

    # Pick only column j → the diagonal: ∂²T/∂x_j²
    H_jj = full_grad[:, j]   # shape (3,)
    print(f'  H_jj = full_grad[:, {j}] = ∂²T/∂{direction}²: {H_jj.detach().numpy()}  (expected: [2. 2. 2.])')

    lap = lap + H_jj.detach()

print('\nFinal Laplacian ∇²T = ∂²T/∂x² + ∂²T/∂y² + ∂²T/∂z²:')
print('  lap:', lap.numpy(), '  expected: [6. 6. 6.]')

Second derivatives (should all be 2.0):

  Differentiating ∂T/∂x again w.r.t. pts:
  full_grad shape: torch.Size([3, 3])  (one row per point, one col per input dim)
  full_grad =
[[2. 0. 0.]
 [2. 0. 0.]
 [2. 0. 0.]]
  H_jj = full_grad[:, 0] = ∂²T/∂x²: [2. 2. 2.]  (expected: [2. 2. 2.])

  Differentiating ∂T/∂y again w.r.t. pts:
  full_grad shape: torch.Size([3, 3])  (one row per point, one col per input dim)
  full_grad =
[[0. 2. 0.]
 [0. 2. 0.]
 [0. 2. 0.]]
  H_jj = full_grad[:, 1] = ∂²T/∂y²: [2. 2. 2.]  (expected: [2. 2. 2.])

  Differentiating ∂T/∂z again w.r.t. pts:
  full_grad shape: torch.Size([3, 3])  (one row per point, one col per input dim)
  full_grad =
[[0. 0. 2.]
 [0. 0. 2.]
 [0. 0. 2.]]
  H_jj = full_grad[:, 2] = ∂²T/∂z²: [2. 2. 2.]  (expected: [2. 2. 2.])

Final Laplacian ∇²T = ∂²T/∂x² + ∂²T/∂y² + ∂²T/∂z²:
  lap: [6. 6. 6.]   expected: [6. 6. 6.]


---
## PART 6 — A function that DOES satisfy Laplace: `T = x - y`

Analytic: `∂²T/∂x² = 0`, `∂²T/∂y² = 0`, `∂²T/∂z² = 0` → `∇²T = 0` ✓

This is what the PINN tries to learn: the physics loss will be 0 for this function.

In [ ]:
pts2 = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
], requires_grad=True)

# T = x - y  (satisfies Laplace: all second derivatives = 0)

T2 = (pts2[:, 0] - pts2[:, 1]).unsqueeze(1)
print('T = x - y:', T2.detach().numpy().flatten())

g2 = torch.autograd.grad(T2.sum(), pts2, create_graph=True)[0]

print('\n∂T/∂x:', g2[:, 0].detach().numpy(), '  (expected: [1. 1. 1.])')
print('∂T/∂y:', g2[:, 1].detach().numpy(), '  (expected: [-1. -1. -1.])')
print('∂T/∂z:', g2[:, 2].detach().numpy(), '  (expected: [0. 0. 0.])')

# g2 is all constants (no grad_fn) because T2 is linear
# → second derivatives are all zero by definition, no autograd needed



T = x - y: [-1. -1. -1.]

∂T/∂x: [1. 1. 1.]   (expected: [1. 1. 1.])
∂T/∂y: [-1. -1. -1.]   (expected: [-1. -1. -1.])
∂T/∂z: [0. 0. 0.]   (expected: [0. 0. 0.])

∇²T (Laplacian): [0. 0. 0.]   (expected: [0. 0. 0.]) ✓
Physics loss = mean(lap²) = 0.0   (expected: 0.0) ✓


---
## PART 7 — Normalisation scaling: why divide by `sx[j]²`?

The network works in **normalised space**: it takes `x̂ = (x - mean) / std` as input.

When we compute `∂²T̂/∂x̂²` we are in normalised units.
To get the physical `∂²T/∂x²` we need the chain rule:

```
∂T/∂x = ∂T̂/∂x̂ × (Y_std / X_std)        ← chain rule, first order
∂²T/∂x² = ∂²T̂/∂x̂² × (Y_std / X_std²)   ← chain rule, second order
```

Since `Y_std > 0`, setting `∇²T = 0` is equivalent to:
```
Σ_j  ∂²T̂/∂x̂_j²  /  X_std[j]²  =  0
```

Let's verify this numerically:

In [ ]:
# Fake normalisation stats (as if coords have different scales)
X_std = torch.tensor([10.0, 5.0, 2.0])   # x varies a lot, z very little
Y_std = torch.tensor([50.0])

# True physical function: T = a*x + b*y + c*z  (linear → Laplacian = 0)
# In normalised coords: T̂ = (T - Y_mean)/Y_std, x̂ = (x - X_mean)/X_std
#
# Normalised equiv: T̂ = a * x̂*X_std[0] / Y_std  + ...
# Second deriv of a linear function = 0, so Laplacian = 0 regardless of scaling.

# Let's use T = x²/X_std[0]² (non-linear, non-zero Laplacian) to see the scaling effect

# Physical: T = (x_phys)²

# Normalised: x_phys = x̂ * X_std[0], so T = (x̂ * X_std[0])²

# T̂ = T / Y_std = (x̂ * X_std[0])² / Y_std

x_hat = torch.tensor([[1.0, 0.5, 0.2],
                       [2.0, 1.0, 0.4]], requires_grad=True)   # normalised coords

# T̂ = x̂² * X_std[0]² / Y_std  (only depends on first coord)

T_hat = (x_hat[:, 0] ** 2) * (X_std[0] ** 2) / Y_std
T_hat = T_hat.unsqueeze(1)
print('T̂ =', T_hat.detach().numpy().flatten())

g_hat = torch.autograd.grad(T_hat.sum(), x_hat, create_graph=True)[0]

lap_normalised = torch.zeros(2)
lap_physical   = torch.zeros(2)

for j in range(3):
    H_jj = torch.autograd.grad(g_hat[:, j].sum(), x_hat, create_graph=True)[0][:, j]
    lap_normalised = lap_normalised + H_jj.detach()
    lap_physical   = lap_physical   + H_jj.detach() / (X_std[j] ** 2)

print('\nLaplacian in normalised space (∂²T̂/∂x̂² + ...):', lap_normalised.numpy())
print('Laplacian in physical space   (/ X_std[j]²)   :', lap_physical.numpy())
print('Physical Laplacian × Y_std                    :', (lap_physical * Y_std).numpy())
print('\n→ The /X_std[j]² converts normalised 2nd-deriv back to physical units')

T̂ = [2. 8.]

Laplacian in normalised space (∂²T̂/∂x̂² + ...): [4. 4.]
Laplacian in physical space   (/ X_std[j]²)   : [0.04 0.04]
Physical Laplacian × Y_std                    : [2. 2.]

→ The /X_std[j]² converts normalised 2nd-deriv back to physical units


---
## PART 8 — Full `physics_loss` with ThermalNet on fake data

Now we put it all together with the real network.

In [ ]:
# ── Fake normalisation stats ──────────────────────────────────────────────────
# In real training these come from the dataset (thermal_norm.npz)
fake_norm = {
    'X_mean': np.zeros(8, dtype=np.float32),
    'X_std':  np.array([50., 50., 500., 1., 1., 1., 1e-5, 1.], dtype=np.float32),
    'Y_mean': np.array([100.], dtype=np.float32),
    'Y_std':  np.array([50.], dtype=np.float32),
}

device = 'cpu'
npt = {k: torch.tensor(v, dtype=torch.float32, device=device) for k, v in fake_norm.items()}

# ── Fake normalised input batch ───────────────────────────────────────────────
N = 6
x_norm = torch.randn(N, 8)   # random normalised inputs

# ── Model (random weights — not trained) ─────────────────────────────────────
model = ThermalNet(hidden=32)   # small for speed
model.eval()

print(f'Input batch x_norm shape: {x_norm.shape}')
print(f'First 3 rows (normalised coords x̂, ŷ, ẑ, ...):')
print(x_norm[:3, :3].numpy().round(3))

In [ ]:
# ── Step 1: clone + enable grad ───────────────────────────────────────────────
x = x_norm.clone().requires_grad_(True)
print('x.requires_grad:', x.requires_grad)   # must be True
print('x.shape:', x.shape)

In [ ]:
# ── Step 2: forward pass ──────────────────────────────────────────────────────
T = model(x)   # (N, 1)  — normalised temperature T̂
print('T (normalised T̂) shape:', T.shape)
print('T values:', T.detach().numpy().flatten().round(4))

In [ ]:
# ── Step 3: std of spatial coords ─────────────────────────────────────────────
sx = npt['X_std'][:3]
print('X_std of [x, y, z]:', sx.numpy())
print('(these are the standard deviations of the raw spatial coordinates in mm)')

In [ ]:
# ── Step 4: first-order spatial gradients ─────────────────────────────────────
g = torch.autograd.grad(T.sum(), x, create_graph=True)[0][:, :3]   # (N, 3)

print('g shape:', g.shape, '  (N=6 points, 3 spatial directions)')
print('\ng[i, j] = ∂T̂_i / ∂x̂_j   (first derivatives, normalised space):')
print(g.detach().numpy().round(4))
print('\nColumns: [∂T̂/∂x̂,  ∂T̂/∂ŷ,  ∂T̂/∂ẑ]')

In [ ]:
# ── Step 5: second-order loop (the diagonal of the Hessian) ───────────────────
lap = torch.zeros(N, device=device)

for j in range(3):
    direction = ['x', 'y', 'z'][j]

    # Differentiate ∂T̂/∂x̂_j once more w.r.t. ALL inputs → shape (N, 8)
    full_second = torch.autograd.grad(g[:, j].sum(), x, create_graph=True)[0]
    print(f'\n--- j={j} ({direction}-direction) ---')
    print(f'  Differentiating g[:, {j}] (= ∂T̂/∂{direction}) w.r.t. x')
    print(f'  full_second shape: {full_second.shape}  (N=6, 8 input dims)')

    # Pick only column j → ∂²T̂/∂x̂_j²  (the diagonal entry)
    H_jj = full_second[:, j]   # shape (N,)
    print(f'  H_jj = full_second[:, {j}] = ∂²T̂/∂{direction}̂²: {H_jj.detach().numpy().round(4)}')
    print(f'  sx[{j}]² = {sx[j].item()**2:.1f}')
    print(f'  H_jj / sx[{j}]² (physical units): {(H_jj / sx[j]**2).detach().numpy().round(6)}')

    lap = lap + H_jj / (sx[j] ** 2)

In [ ]:
# ── Step 6: final loss ────────────────────────────────────────────────────────
print('\nFinal Laplacian per point (∇²T in physical units):')
print(lap.detach().numpy().round(6))
print('\nlap² per point:')
print((lap ** 2).detach().numpy().round(8))

loss = torch.mean(lap ** 2)
print(f'\nPhysics loss = mean(lap²) = {loss.item():.6f}')
print('\n(This is large because the network has random weights — not trained yet.)')
print('After training, this should approach 0.0')

---
## PART 9 — Summary: the full function at a glance

```python
def physics_loss(model, x_norm, npt):

    x = x_norm.clone().requires_grad_(True)   # enable tracking
    T = model(x)                              # forward pass → T̂  (N,1)

    sx = npt['X_std'][:3]                     # std of x,y,z

    # 1st derivatives: g[i,j] = ∂T̂_i/∂x̂_j
    g = autograd.grad(T.sum(), x, create_graph=True)[0][:, :3]

    lap = zeros(N)
    for j in 0,1,2:
        # 2nd derivative diagonal: ∂²T̂_i/∂x̂_j²
        H_jj = autograd.grad(g[:,j].sum(), x, create_graph=True)[0][:, j]
        lap  = lap + H_jj / sx[j]²            # scale to physical

    # lap[i] ≈ ∇²T at point i  (should be 0 everywhere)
    return mean(lap²)
```

| Step | What it computes | Shape |
|------|-----------------|-------|
| `model(x)` | T̂ — normalised temperature | (N, 1) |
| `grad(T.sum(), x)[0][:, :3]` | ∂T̂/∂x̂_j — 1st spatial derivatives | (N, 3) |
| `grad(g[:,j].sum(), x)[0][:, j]` | ∂²T̂/∂x̂_j² — 2nd derivative, diagonal only | (N,) |
| `H_jj / sx[j]²` | same in physical units | (N,) |
| `mean(lap²)` | mean squared Laplacian residual | scalar |